In [1]:
# Downloading the shakesphere dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-09-12 22:13:31--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-09-12 22:13:32 (22.1 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
# let us read and inspect it
with open('input.txt', 'r' , encoding = 'utf-8') as f:
    text = f.read()

In [3]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [4]:
# let us look at the first 1k characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
# finding out the unique characters in the set
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
# let us develop some strategy to tokensize the text - one of out of many tokenization techniques and very simple
# create mapping
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hi there!"))
print(decode(encode("hi there!")))

[46, 47, 1, 58, 46, 43, 56, 43, 2]
hi there!


In [7]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape , data.dtype)
print(data[:1000]) # lets see how 1st 1k charcters look when rep.as tokens

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

The bigger the size of each token lesser the size of vocab visavis

In [8]:
# splitting up the dataset
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

We train the transformer on a tiny little chunk of the dataset not everyting at once


In [9]:
# we put plus one, to make the amount of learning(8 indv examples packed in) equal to the set block size
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [10]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when in put is {context} the target:{target}")

when in put is tensor([18]) the target:47
when in put is tensor([18, 47]) the target:56
when in put is tensor([18, 47, 56]) the target:57
when in put is tensor([18, 47, 56, 57]) the target:58
when in put is tensor([18, 47, 56, 57, 58]) the target:1
when in put is tensor([18, 47, 56, 57, 58,  1]) the target:15
when in put is tensor([18, 47, 56, 57, 58,  1, 15]) the target:47
when in put is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target:58


In [11]:
torch.manual_seed(1337) # cause it randomly picks up batch_size amount of sequences
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # this is the maximum context length of the predictions

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data # this gives us the data tensor
    # remember train data is also a tensor made a while ago
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

These are 32 input sequences packed into this batch

In [12]:
print(xb) # this is what we are going to feed into the transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [13]:
# we will feed it to the simplest language model 
# which is the bigram language model
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module): # we are predicting what comes next based on the individual identity of a single token
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from this lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    def forward(self, idx, targets =None):
        logits = self.token_embedding_table(idx) #(Batch, time(context length/block size),length of each embedding)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B*T)
            # in pytorch documentation 
            ''' it expects the dimensions of the logits to be mini batch * channels(which are the dimensions of the tokens)*t
            so we merge them up into 2d tensor, to make those embedding dimensions go into the 2nd dimension of the logits'''
            loss = F.cross_entropy(logits,targets) # simiarly aligning the targets
        return logits , loss
    def generate(self, idx, max_new_tokens):
        # idx is (B,T) array of indices in the current context
        for _ in range(max_new_tokens):
            #get the predictions
            logits,loss = self(idx)
            # we will grab out only the last time step, as this is bigram model only-means we need only the character before what we wanna predict
            logits = logits[:,-1,:]
            # apply softmax to get the probs
            probs = F.softmax(logits, dim=-1) # meas what shape we are giving that shape you outpt
            #sample from the distribution
            idx_next = torch.multinomial(probs, num_samples = 1)# this is like a spin wheel with probabailities
            idx = torch.cat((idx, idx_next), dim=1) #(B, T+1)
        return idx
                        
m = BigramLanguageModel(vocab_size)
logits,loss = m(xb, yb)
print(logits.shape)
print(logits)
print(loss) # which should be around 1/65 , expected



torch.Size([32, 65])
tensor([[-1.5101, -0.0948,  1.0927,  ..., -0.6126, -0.6597,  0.7624],
        [ 0.3323, -0.0872, -0.7470,  ..., -0.6716, -0.9572, -0.9594],
        [ 0.2475, -0.6349, -1.2909,  ...,  1.3064, -0.2256, -1.8305],
        ...,
        [-2.1910, -0.7574,  1.9656,  ..., -0.3580,  0.8585, -0.6161],
        [ 0.5978, -0.0514, -0.0646,  ..., -1.4649, -2.0555,  1.8275],
        [-0.6787,  0.8662, -1.6433,  ...,  2.3671, -0.7775, -0.2586]],
       grad_fn=<ViewBackward0>)
tensor(4.8786, grad_fn=<NllLossBackward0>)


In [14]:
# now lets do some generation with the bigram model we created
print(decode(m.generate(idx=torch.zeros((1,1), dtype=torch.long),max_new_tokens=100)[0].tolist()))


SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [15]:
# create a pytorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
# it will take the gradients and update the params acc to the grdients

In [16]:
batch_size = 32
for steps in range(10000): 

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
print(loss.item())


2.382369041442871


In [17]:
# we are getting smtg like reasonable-ish
print(decode(m.generate(idx=torch.zeros((1,1), dtype=torch.long),max_new_tokens=500)[0].tolist()))


lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulseecherd d o blllando;LUCEO, oraingofof win!
RIfans picspeserer hee tha,
TOFonk? me ain ckntoty ded. bo'llll st ta d:
ELIS me hurf lal y, ma dus pe athouo
BEY:! Indy; by s afreanoo adicererupa anse tecorro llaus a!
OLeneerithesinthengove fal amas trr
TI ar I t, mes, n IUSt my w, fredeeyove
THek' merer, dd
We ntem lud engitheso; cer ize helorowaginte the?
Thak orblyoruldvicee chot, p,
Bealivolde Th li


some layers have diffrent behaviour in training/eval time, keep note of that
plus
estimate_loss -> averages up the loss over multiple batches, we will get avg for both splits
@torch.no_grad() -> we will never need gradients over here, dont keep track

In [18]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


### Trick in self attention

In [19]:
# considering this toy example
torch.manual_seed(1337)
B,T,C = 4,8, 2
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [20]:
# we want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev,0)

Here we are averaging stuff and putting it into xbow and saving values across time(making them talk to a trivial way basically). so x[b,t,c] is a normal embedding which is formed and xprev is what adds up that particular batch ka embeddings , and we take avg of xprev(in the same incremental fashion) in time dimension as planned above and put it in xbow

In [21]:
x[0] # first batch of random x

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [22]:
xbow[0] # first batch of avergaed x

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

To do the same incremental averaging, we propose having this manipulative matrix a and do it indirectly

In [23]:
torch.manual_seed(42)
# a = torch.ones(3,3)
a = torch.tril(torch.ones(3,3)) # we are forming the lower traingular matrix out of this
a = a / torch.sum(a, 1, keepdim=True) # sum a in the dimension of 1 and keep the dimensions for braodcasting
b = torch.randint(0,10,(3,2)).float()
c= a @ b # matrix multiplication is being performed
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


So by manipulating the a like how we did above , we can find averages of the matrix in the incremental fashion like we just did above!

In [24]:
# ver-2 , using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1, keepdim=True) # dim needed for future brodcasting
xbow2 = wei @ x # (B,T,T) @ (B,T,C) -> BTC, in wei B is created as its needed and matrix mulplication happens
# checking sanity 
print(torch.allclose(xbow, xbow2, atol=1e-7))

True


In [25]:
xbow2[0]  # this looks same as xbow

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

So we are pulling all these stunts to find weighted aggregates which is gonna help us in future, plus we make sure that a particular token gets info from only the tokens perceiving it

In [26]:
tril=torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T)) # its begins with zero , but they are going to be dependent , they will look at each other anol
# and it tells us how much from each token in the past do you wanna aggregate on
wei=wei.masked_fill(tril==0,float('-inf')) # cause they arent supposed to see the future
wei=F.softmax(wei,dim=-1)#self attention trick
xbow3=wei@x #(B,T,T) @ (B,T,C) → (B,T,C)
torch.allclose(xbow,xbow3)

False

**We can do weighted aggregations by using matrix multiplication of lower triangular fashion, the weights say how much of each elements fuses into this aggregation**

In [27]:
# In the script bigram_with_attn.py ,there was addition of the positional encoding and also added intermediate embedding layer rather than a normal look up
# which basically means creating a level of indirection, which gives the token embeddings
# and we add a linear layer which gives us logits from the token embeddings
# self.lm_head = nn.Linear(n_embd , vocab_size)
# logits = self.lm_head() this give logits , which aint B,T,C but its B,T,vocab_size

In [28]:
# self.position_embedding_table = nn.Embedding(block_size , n_embed)
# we will have a second position embedding table over here, so each position will get an embedding vector

In [29]:
'''
tok_emb = self.token_embedding_table(idx)
pos_emb = self. positon_embedding_table(torch.arange(T, device=device)) # (T,C) -> broadcasting will happen here
x = tok_emb + pos_emb # (B,T,C)
'''

'\ntok_emb = self.token_embedding_table(idx)\npos_emb = self. positon_embedding_table(torch.arange(T, device=device)) # (T,C) -> broadcasting will happen here\nx = tok_emb + pos_emb # (B,T,C)\n'

In [32]:
# ver-4 , here we shall implement simple self-attention
# what were doing till was just averaging the past tokens and current one, now lets consider the affinities among and do it
torch.manual_seed(1337)
B,T,C = 4, 8, 32 # batch, time, channels
x = torch.randn(B,T,C) # creating random embedding initializations

# every token emits , q,k,  q-what do i need , k-what do i contain, v-this is what i can deliver to you if you like me(if you you find me interesting here's what i will communicate to you)
# my query dot pdts with all the tokens and that dotpdts we will put in wei
head_size = 16
# we will change the  whatever number of dimensions in embedding vector into 16 dims for doing our attn thing
key = nn.Linear(C,head_size, bias = False)
query = nn.Linear(C,head_size, bias =False)
value = nn.Linear(C,head_size, bias =False)
k = key(x)
q = query(x)
wei = q @ k.transpose(-2,-1) #(B,T,16) @ (B,16,T) → (B,T,T) , and in that T,T table only we have affinites baked into
# now when we do aggreation this time, it will have insights from the data

tril=torch.tril(torch.ones(T,T))
wei=wei.masked_fill(tril==0,float('-inf'))
wei=F.softmax(wei,dim=-1)#self attention trick, softmax is to give nice distirbution
#out=wei@x #(B,T,T) @ (B,T,C) → (B,T,C)

out.shape
# in this case we wont aggrete over the tokens, rather over its v-value
v=value(x)
out=wei@v

In [33]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

In [ ]:
'''
1. attention is a communication mechanism
2. over here there is no notion of space, they r simply nodes with vectors doing their thing
3. elements across the are batch dimnesion are independent and never talk to each other for any reason
4. since this is autoregressive we dont make past talk to future but for other reasons like sentiment analysis or smtg you can make everyone talk to everyone
5. called 'self' cause q, k,v everything comes from the same src
cross attention happens when we grab info from different set of nodes where we wanna learn from
'''

In [34]:
# they proposed scaled attn with sqrt(dk(head size)) which is like an important normalisation to have
# we need fairly diffused numbers to initialize wei else
# if it takes very high and low -ve values softmax will give one hot like stuff
# softmax sharpens towards the max like rich gets richer
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [35]:
k.var()

tensor(1.0449)

In [36]:
q.var()

tensor(1.0700)

In [37]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [38]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

See how softmax gets way too peaky! we dont wanna aggregate info from a single node
So scaling is used to control variance at initialization

In PyTorch, register_buffer is a method used to store a tensor that is part of a module's state but not a learnable parameter


reduced lr a bit cause self attn cant tolerate high learning rates

In [39]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

In [40]:
x[:,0].mean(), x[:,0].std() # mean,std of one feature across all batch inputs

(tensor(0.1469), tensor(0.8803))

In [41]:
x[0,:].mean(), x[0,:].std() # mean,std of a single input from the batch, of its features

(tensor(-9.5367e-09), tensor(1.0000))

French to English translation example:

<--------- ENCODE ------------------><--------------- DECODE ----------------->  
J'accepte la grande aventure d'être moi <START> I accept the great adventure of being myself<END>